### ============================================
### PARTIE 1 : EXTRACTION CONCEPTS MÉDICAUX
### ============================================

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print(" EXTRACTION CONCEPTS MÉDICAUX")
print("=" * 60)

 EXTRACTION CONCEPTS MÉDICAUX


### --------------------------------------------
### 1. CHARGER DONNÉES
### --------------------------------------------


In [3]:

print("\n Chargement des données...")
train_df = pd.read_csv("../data/processed/train_data.csv")
val_df = pd.read_csv("../data/processed/val_data.csv")
test_df = pd.read_csv("../data/processed/test_data.csv")

print(f" Train : {len(train_df):,} notes")
print(f" Val : {len(val_df):,} notes")
print(f" Test : {len(test_df):,} notes")


 Chargement des données...
 Train : 59,892 notes
 Val : 11,636 notes
 Test : 12,164 notes



### --------------------------------------------
### 2. NETTOYAGE TEXTE
### --------------------------------------------


In [4]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\n", " ", text)

    return text.strip()


for df in [train_df, val_df, test_df]:
    df["TEXT_CLEAN"] = df["TEXT_CLEAN"].apply(clean_text)

### --------------------------------------------
### 3. DICTIONNAIRE MÉDICAL
### --------------------------------------------


In [5]:
print("\n Configuration extraction concepts...")

MEDICAL_TERMS = {

    "diseases": [
        "copd","pneumonia","sepsis","infection","infarction","hypertension",
        "diabetes","cardiomyopathy","arrhythmia","embolism","stroke","cancer",
        "cirrhosis","hepatitis","nephropathy","anemia","thrombosis","ischemia",
        "edema","decompensation","exacerbation","failure"
    ],

    "symptoms": [
        "pain","dyspnea","fever","hypoxia","tachycardia","bradycardia",
        "hypotension","nausea","vomiting","diarrhea","confusion",
        "weakness","fatigue","shortness of breath","chest pain",
        "abdominal pain","headache","dizziness"
    ],

    "procedures": [
        "intubation","ventilation","dialysis","surgery","catheter",
        "transfusion","biopsy","ct scan","mri","x-ray",
        "echocardiogram","ekg","bronchoscopy","endoscopy"
    ],

    "medications": [
        "antibiotic","insulin","heparin","warfarin","aspirin","statin",
        "beta blocker","ace inhibitor","diuretic","steroid","prednisone",
        "morphine","fentanyl","vancomycin","levofloxacin","metformin",
        "lisinopril","metoprolol","furosemide","lasix"
    ],

    "severity": [
        "acute","chronic","severe","critical","mild","moderate",
        "emergency","urgent","stable","unstable","life-threatening",
        "terminal","palliative","icu","intensive care"
    ],

    "labs": [
        "creatinine","glucose","hemoglobin","platelets","wbc",
        "sodium","potassium","lactate","troponin","bilirubin",
        "blood pressure","heart rate","respiratory rate","oxygen saturation"
    ]
}
# Précompiler regex (beaucoup plus rapide)
REGEX_TERMS = {
    category: [re.compile(r"\b" + re.escape(term) + r"\b") for term in terms]
    for category, terms in MEDICAL_TERMS.items()
}


 Configuration extraction concepts...


### --------------------------------------------
### 4. EXTRACTION CONCEPTS
### --------------------------------------------


In [6]:

def extract_medical_concepts(text, max_per_category=5):

    if pd.isna(text) or str(text).strip() == "":
        return {cat: [] for cat in MEDICAL_TERMS.keys()}

    text = str(text).lower()

    concepts = {}

    for category, patterns in REGEX_TERMS.items():

        found = []

        for pattern, term in zip(patterns, MEDICAL_TERMS[category]):

            if pattern.search(text):
                found.append(term)

            if len(found) >= max_per_category:
                break

        concepts[category] = found

    return concepts

### --------------------------------------------
### 5. EXTRACTION SUR DATASET
### --------------------------------------------

In [7]:

def extract_for_dataset(df, name):

    print(f"\n Extraction {name}")

    concepts_list = []

    for text in tqdm(df["TEXT_CLEAN"], total=len(df)):
        concepts = extract_medical_concepts(text)
        concepts_list.append(concepts)

    return concepts_list


train_concepts = extract_for_dataset(train_df, "TRAIN")
val_concepts = extract_for_dataset(val_df, "VAL")
test_concepts = extract_for_dataset(test_df, "TEST")

print("\n Extraction terminée")


 Extraction TRAIN


  0%|          | 0/59892 [00:00<?, ?it/s]


 Extraction VAL


  0%|          | 0/11636 [00:00<?, ?it/s]


 Extraction TEST


  0%|          | 0/12164 [00:00<?, ?it/s]


 Extraction terminée


### --------------------------------------------
### 6. AJOUT AUX DATAFRAMES
### --------------------------------------------

In [8]:

def concepts_to_string(concepts):

    parts = []

    for category, terms in concepts.items():

        if terms:
            parts.append(f"{category}: {', '.join(terms)}")

    return " | ".join(parts) if parts else "none"


def add_concepts(df, concepts):

    df["extracted_diseases"] = [c["diseases"] for c in concepts]
    df["extracted_symptoms"] = [c["symptoms"] for c in concepts]
    df["extracted_procedures"] = [c["procedures"] for c in concepts]
    df["extracted_medications"] = [c["medications"] for c in concepts]
    df["extracted_severity"] = [c["severity"] for c in concepts]
    df["extracted_labs"] = [c["labs"] for c in concepts]

    df["all_concepts"] = [concepts_to_string(c) for c in concepts]


add_concepts(train_df, train_concepts)
add_concepts(val_df, val_concepts)
add_concepts(test_df, test_concepts)

print(" Concepts ajoutés aux datasets")

 Concepts ajoutés aux datasets


### --------------------------------------------
### 7. CRÉATION PROMPTS ENRICHIS
### --------------------------------------------

In [9]:
print("\n Création prompts Clinical-T5")

def create_enriched_prompt(row):

    diseases = ", ".join(row["extracted_diseases"][:3]) or "none"
    symptoms = ", ".join(row["extracted_symptoms"][:3]) or "none"
    procedures = ", ".join(row["extracted_procedures"][:2]) or "none"
    severity = ", ".join(row["extracted_severity"][:2]) or "none"
    labs = ", ".join(row["extracted_labs"][:3]) or "none"

    diagnoses = str(row["diagnosis_labels"])[:120]
    text = str(row["TEXT_CLEAN"])[:800]

    prompt = f"""
You are a medical AI assistant.

Write a concise clinical summary for a {row['persona_target']}.

Include:
- main diagnosis
- key symptoms
- important lab results
- treatments or procedures
- patient condition

Medical concepts extracted:
Diseases: {diseases}
Symptoms: {symptoms}
Procedures: {procedures}
Labs: {labs}
Severity: {severity}

Diagnoses: {diagnoses}

Clinical note:
{text}
"""

    return prompt.strip()


 Création prompts Clinical-T5


### --------------------------------------------
###  8. TARGET SUMMARY
### --------------------------------------------

In [10]:
def create_target(row):

    text = str(row["TEXT_CLEAN"])

    sentences = re.split(r"[.]", text)

    sentences = [
        s.strip() for s in sentences
        if len(s.strip()) > 30
    ]

    summary = ". ".join(sentences[:3])

    if summary and not summary.endswith("."):
        summary += "."

    return summary[:500]


for df in [train_df, val_df, test_df]:

    df["prompt_enriched"] = df.apply(create_enriched_prompt, axis=1)
    df["target"] = df.apply(create_target, axis=1)

print(" Prompts créés")

 Prompts créés


### --------------------------------------------
###  9. STATISTIQUES
### --------------------------------------------

In [11]:
print("\n Statistiques TRAIN")

print(
    "Notes avec maladies:",
    (train_df["extracted_diseases"].apply(len) > 0).sum()
)

print(
    "Notes avec symptômes:",
    (train_df["extracted_symptoms"].apply(len) > 0).sum()
)

print(
    "Notes avec médicaments:",
    (train_df["extracted_medications"].apply(len) > 0).sum()
)

print("\n Exemple concepts")
print(train_df["all_concepts"].iloc[0])

print("\n Exemple prompt")
print(train_df["prompt_enriched"].iloc[0][:600])



 Statistiques TRAIN
Notes avec maladies: 23810
Notes avec symptômes: 21807
Notes avec médicaments: 15758

 Exemple concepts
diseases: copd, hypertension, edema, exacerbation, failure | symptoms: pain, fever, hypoxia, shortness of breath, chest pain | procedures: intubation, surgery, bronchoscopy | medications: heparin, vancomycin, lasix | severity: severe, icu | labs: wbc, sodium

 Exemple prompt
You are a medical AI assistant.

Write a concise clinical summary for a Généraliste.

Include:
- main diagnosis
- key symptoms
- important lab results
- treatments or procedures
- patient condition

Medical concepts extracted:
Diseases: copd, hypertension, edema
Symptoms: pain, fever, hypoxia
Procedures: intubation, surgery
Labs: wbc, sodium
Severity: severe, icu

Diagnoses: Chr airway obstruct NEC | Acidosis | Ac DVT/embl low ext NOS | Diaphragmatic hernia

Clinical note:
Admission Date: Discharge Date: Service: CARDIOTHORACIC Allergies: Amlodipine Attending: Chief Complaint: 81 yo F smoker 

### --------------------------------------------
### 10. SAUVEGARDE
### --------------------------------------------

In [12]:
print("\n Sauvegarde datasets enrichis")

train_df.to_csv("../data/curated/train_enriched.csv", index=False)
val_df.to_csv("../data/curated/val_enriched.csv", index=False)
test_df.to_csv("../data/curated/test_enriched.csv", index=False)

print("\n Fichiers créés")

print("\n" + "=" * 60)
print(" DATASETS PRÊTS POUR CLINICAL-T5")
print("=" * 60)


 Sauvegarde datasets enrichis

 Fichiers créés

 DATASETS PRÊTS POUR CLINICAL-T5
